In [ ]:
pip install scikit-surprise

In [ ]:
!pip install numpy==1.26.4
!pip install scikit-surprise


In [ ]:
import pandas as pd

data = pd.read_csv("/content/cleaned_ecommerce_dataset.csv")

print(data.head())
print("Dataset shape:", data.shape)


                            user_id                           item_id  \
0  98758d88bf4b8eef1372ddee45d63178  fe59a1e006df3ac42bf0ceb876d70969   
1  87ae4c644c15d9c6b6f826dfec33b340  e19ddcc85537b41f22116c8d5425ef46   
2  2e875ea57961ad115cec13fef0920ae6  880be32f4db1d9f6e2bec38fb6ac23ab   
3  3c857a6f7828bfb70fb712e2393cfd1b  1f9799a175f50c9fa725984775cac5c5   
4  3c857a6f7828bfb70fb712e2393cfd1b  13944d17b257432717fd260e69853140   

                 category   price  freight_value  customer_city  \
0  informatica_acessorios  809.10          44.29   campo alegre   
1        moveis_decoracao   29.99          15.10  volta redonda   
2              brinquedos   44.90           7.16   porto alegre   
3         cama_mesa_banho   59.90           9.94      sao paulo   
4         cama_mesa_banho   59.90           9.94      sao paulo   

  customer_state  interaction  purchase_count  
0             AL            1               1  
1             RJ            1               1  
2             

In [ ]:
from surprise import Dataset, Reader


In [ ]:
# Define rating scale using purchase_count
reader = Reader(
    rating_scale=(1, int(data['purchase_count'].max()))
)

dataset = Dataset.load_from_df(
    data[['user_id', 'item_id', 'purchase_count']],
    reader
)

In [ ]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42
)

In [ ]:
from surprise import SVD, accuracy

In [ ]:
svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("SVD model trained successfully")


SVD model trained successfully


In [ ]:
predictions = svd_model.test(testset)

print("Baseline Model Performance:")
accuracy.rmse(predictions)
accuracy.mae(predictions)


Baseline Model Performance:
RMSE: 0.0828
MAE:  0.0241


0.02410594187451262

In [ ]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=5, threshold=1):
    user_est_true = defaultdict(list)

    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions, recalls = {}, {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]

        relevant_items = sum(true_r >= threshold for _, true_r in user_ratings)
        recommended_relevant = sum(
            (est >= threshold and true_r >= threshold)
            for est, true_r in top_k
        )

        precisions[uid] = recommended_relevant / k if k else 0
        recalls[uid] = recommended_relevant / relevant_items if relevant_items else 0

    precision = sum(precisions.values()) / len(precisions)
    recall = sum(recalls.values()) / len(recalls)

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    return precision, recall, f1

In [ ]:
p, r, f1 = precision_recall_at_k(predictions, k=5)

print(f"Precision@5: {p:.4f}")
print(f"Recall@5: {r:.4f}")
print(f"F1-Score@5: {f1:.4f}")

Precision@5: 0.2176
Recall@5: 1.0000
F1-Score@5: 0.3574


In [ ]:
print("Min purchase count:", data['purchase_count'].min())
print("Max purchase count:", data['purchase_count'].max())
print("Unique purchase counts and their frequencies:\n", data['purchase_count'].value_counts().sort_index())

Min purchase count: 1
Max purchase count: 3
Unique purchase counts and their frequencies:
 purchase_count
1    5599
2      21
3       3
Name: count, dtype: int64


In [ ]:
svd_refined = SVD(
    n_factors=100,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.01,
    random_state=42
)

svd_refined.fit(trainset)


In [ ]:
refined_predictions = svd_refined.test(testset)

print("\nRefined Model Performance:")
accuracy.rmse(refined_predictions)
accuracy.mae(refined_predictions)



Refined Model Performance:
RMSE: 0.0882
MAE:  0.0292


0.029164232166688987

In [ ]:
p_r, r_r, f1_r = precision_recall_at_k(refined_predictions, k=5)

print(f"Refined Precision@5: {p_r:.4f}")
print(f"Refined Recall@5: {r_r:.4f}")
print(f"Refined F1-Score@5: {f1_r:.4f}")


Refined Precision@5: 0.2176
Refined Recall@5: 1.0000
Refined F1-Score@5: 0.3574


In [ ]:
def recommend_products_svd(model, data, user_id, N=5):
    all_items = data['item_id'].unique()

    predictions = [
        (item, model.predict(user_id, item).est)
        for item in all_items
    ]

    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:N]


In [ ]:
sample_user = data['user_id'].iloc[0]

recommendations = recommend_products_svd(
    svd_refined, data, sample_user, N=5
)

print("\nTop-5 Recommendations:")
for item, score in recommendations:
    print(item, "→", round(score, 2))



Top-5 Recommendations:
813329d52f377306fa1614ca9a962cd5 → 1.36
f0b543161e745b6c80a79c368db167a5 → 1.36
8c591ab0ca519558779df02023177f44 → 1.3
d0fe4295267f15ccaceac4fb233d8c9a → 1.26
26a5af09b51d4e5f17609f6c9ae7f50f → 1.26
